In [ ]:
import os
import sys
from pathlib import Path
sys.path.append(os.path.join(Path().resolve(), '..'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from src.module.dmd import DMD

from cycler import cycler
from matplotlib.animation import FuncAnimation

plt.rcParams['text.color'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 15
plt.rcParams['axes.titlecolor'] = 'white'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams["axes.facecolor"] = "#191919"
plt.rcParams["axes.edgecolor"] = "white"
plt.rcParams["figure.facecolor"] = "#191919"
plt.rcParams["figure.edgecolor"] = "white"
plt.rcParams["legend.facecolor"] ="dimgray"
plt.rcParams["legend.labelcolor"] ="white"
plt.rcParams['axes.prop_cycle'] = cycler('color', ['#8dd3c7', '#feffb3', '#bfbbd9', '#fa8174', '#81b1d2', '#fdb462', '#b3de69', '#bc82bd', '#ccebc4', '#ffed6f'])

## Synthetic Data

In [ ]:
t = np.arange(0, 10, 0.01)
dt = t[2] - t[1]

OBSERVASION_TIME = 1
trend = 10 * (t/7 - OBSERVASION_TIME) ** 2
periodic1 = np.sin(10 * 2 * np.pi * t/10) / np.exp(-2 * t/10)
periodic2 = np.sin(5 * 2 * np.pi * t)
np.random.seed(123)
noise = 1.5 * (np.random.rand(len(t)) - 0.5)

x1 = np.exp(t/5) * np.sin(2*np.pi*t)
x2 = np.cos(3*np.pi*t)
x3 = 5*np.exp(-t/5)
x4 = trend + periodic1 + periodic2 + noise

plt.figure(figsize=(20, 4))
plt.plot(t, x1, label='x1')
plt.plot(t, x2, label='x2')
plt.plot(t, x3, label='x3')
plt.plot(t, x4, label='x3')

## Initialize

In [ ]:
TRAIN = 700
h = 4

In [ ]:
data = np.array([x1, x2, x3, x4])
model = DMD(h=h, thresh=0.999, dt=0.01)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X[:, :TRAIN - h], Y[:, :TRAIN - h])

pred_fit = model.predict(X[:, 0], show=True)

## Update

In [ ]:
old_timestep = model.timestep
for xt, yt in zip(X[:, TRAIN - h: 700].T, Y[:, TRAIN - h: 700].T):
    model.learn_one(xt.reshape(-1, 1), yt.reshape(-1, 1))
print(f"timestamp: {old_timestep} ===> {model.timestep}")

pred_stream = model.predict(X[:, 0], show=True)

## Motion Capture Data

In [ ]:
mocap = pd.read_csv('../data/mocap/20_01.amc.4d', delim_whitespace=True).to_numpy()

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(mocap)

In [ ]:
data = mocap[:730].T
model = DMD(h=4, thresh=0.9999, dt=1)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X, Y)
pred = model.predict(X[:, 0].T, width=(0, 200), show=False)

In [ ]:
data = mocap[:200].T
model = DMD(h=4, thresh=0.9999, dt=1)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X, Y)
pred1 = model.predict(X[:, 0].T, width=(0, 200), show=False)

In [ ]:
data = mocap[200: 350].T
model = DMD(h=4, thresh=0.9999, dt=1)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X, Y)
pred2 = model.predict(X[:, 0], width=(0, 150), show=False)

In [ ]:
data = mocap[350: 550].T
model = DMD(h=4, thresh=0.999, dt=1)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X, Y)
pred3 = model.predict(X[:, 0], width=(0, 200), show=False)

In [ ]:
data = mocap[550: 730].T
model = DMD(h=4, thresh=0.999, dt=1)
X, Y = model.create_variables(data)

print("----------------- Input Data -----------------")
print("  < raw data >")
print(f"  ・dimension:\t{data.shape[0]}")
print(f"  ・length:\t{data.shape[1]}")
print("----------------------------------------------")
print("--------------- Preprocessing ----------------")
print("  < \u03a8 >")
print(f"  ・dimension:\t{X.shape[0]}")
print(f"  ・length:\t{X.shape[1] + 1}")
print("----------------------------------------------")
model.fit(X, Y)
pred4 = model.predict(X[:, 0], width=(0, 200), show=False)

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(np.hstack(np.real(pred[0])))
plt.plot(np.hstack(np.real(pred[4])))
plt.plot(np.hstack(np.real(pred[8])))
plt.plot(np.hstack(np.real(pred[12])))

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(np.hstack([np.real(pred1[0]), np.real(pred2[0]), np.real(pred3[0]), np.real(pred4[0])]))
plt.plot(np.hstack([np.real(pred1[4]), np.real(pred2[4]), np.real(pred3[4]), np.real(pred4[4])]))
plt.plot(np.hstack([np.real(pred1[8]), np.real(pred2[8]), np.real(pred3[8]), np.real(pred4[8])]))
plt.plot(np.hstack([np.real(pred1[12]), np.real(pred2[12]), np.real(pred3[12]), np.real(pred4[12])]))

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(mocap[:730])